In [1]:
# # Row Based File Formats
# CSV --> 1GB
# AVRO --> ~600MB

# # Column based file formats
# parquet --> ~450MB
# ORC ~900MB

#CSV file of 50MB --> When Converted into parquet file it will reduce to --> around 20MB



In [6]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Retail').getOrCreate()
csv_df = spark.read.csv('retail_sales_dataset.csv', header = True)
csv_df.count()

5000

In [11]:
# coalesce(1) --> Data will be overwritten into one single file ( you will see the 'SUCCESS' msg)
# Compression will be applied
# Default File format of Spark  is Parquet
csv_df.coalesce(1).write.mode('overwrite').parquet('parquet_sales_data')

In [12]:
# ORC File Format
# You Can Download and check the size of the file
csv_df.coalesce(1).write.mode('overwrite').orc('orc_sales_data')

In [ ]:
# avro file format is not inbuilt in SparkSession, so It is requires to import external packages for writing avro file format


In [24]:
#Joins: --> combining

emp_columns =['id','name', 'dept_id']
emp_data = [(1, 'Gayatri', 1), (2, 'Ram', 2), (3, 'Priya', 4)]
dept_columns = ['id', 'name', 'revenue']
dept_data = [(1, 'IT', 230098456), (2, 'Finance', 78374874), (3, 'HR', 3435543634),]
emp_df = spark.createDataFrame(emp_data, emp_columns)
dept_df = spark.createDataFrame(dept_data, dept_columns)
emp_df.show()
dept_df.show()

+---+-------+-------+
| id|   name|dept_id|
+---+-------+-------+
|  1|Gayatri|      1|
|  2|    Ram|      2|
|  3|  Priya|      4|
+---+-------+-------+

+---+-------+----------+
| id|   name|   revenue|
+---+-------+----------+
|  1|     IT| 230098456|
|  2|Finance|  78374874|
|  3|     HR|3435543634|
+---+-------+----------+



In [25]:
# Default Join Type is INNER JOIN
# Inner Join: dept, emp ==> joining column, condition
                    # emp ==> dept_id
                    # dept ==> id
                    # condition => dept_id == id --> Matching Record
# inner_join_df = emp_df.join(dept_df, emp_df.dept_id == dept_df.id,) ----> Default Inner join
inner_join_df = emp_df.join(dept_df, emp_df.dept_id == dept_df.id, 'inner')
inner_join_df.show()


+---+-------+-------+---+-------+---------+
| id|   name|dept_id| id|   name|  revenue|
+---+-------+-------+---+-------+---------+
|  1|Gayatri|      1|  1|     IT|230098456|
|  2|    Ram|      2|  2|Finance| 78374874|
+---+-------+-------+---+-------+---------+



In [26]:
# Left Join --> Matching Records from right side and all records from left side
left_join_df = emp_df.join(dept_df, emp_df.dept_id == dept_df.id, 'left')
left_join_df.show()

+---+-------+-------+----+-------+---------+
| id|   name|dept_id|  id|   name|  revenue|
+---+-------+-------+----+-------+---------+
|  1|Gayatri|      1|   1|     IT|230098456|
|  2|    Ram|      2|   2|Finance| 78374874|
|  3|  Priya|      4|NULL|   NULL|     NULL|
+---+-------+-------+----+-------+---------+



In [27]:
# Right Join --> Matching records from left side, and all records from left join
right_join_df = emp_df.join(dept_df, emp_df.dept_id == dept_df.id, 'right')
right_join_df.show()

+----+-------+-------+---+-------+----------+
|  id|   name|dept_id| id|   name|   revenue|
+----+-------+-------+---+-------+----------+
|   1|Gayatri|      1|  1|     IT| 230098456|
|NULL|   NULL|   NULL|  3|     HR|3435543634|
|   2|    Ram|      2|  2|Finance|  78374874|
+----+-------+-------+---+-------+----------+



In [28]:
# Full Join --> Matching records + Right side records + left side records
full_join_df = emp_df.join(dept_df, emp_df.dept_id == dept_df.id, 'full')
full_join_df.show()

+----+-------+-------+----+-------+----------+
|  id|   name|dept_id|  id|   name|   revenue|
+----+-------+-------+----+-------+----------+
|   1|Gayatri|      1|   1|     IT| 230098456|
|   2|    Ram|      2|   2|Finance|  78374874|
|NULL|   NULL|   NULL|   3|     HR|3435543634|
|   3|  Priya|      4|NULL|   NULL|      NULL|
+----+-------+-------+----+-------+----------+



In [29]:
# Cross Join --> each and every value matching from left and right side table
emp_df.crossJoin(dept_df).show()


+---+-------+-------+---+-------+----------+
| id|   name|dept_id| id|   name|   revenue|
+---+-------+-------+---+-------+----------+
|  1|Gayatri|      1|  1|     IT| 230098456|
|  1|Gayatri|      1|  2|Finance|  78374874|
|  1|Gayatri|      1|  3|     HR|3435543634|
|  2|    Ram|      2|  1|     IT| 230098456|
|  3|  Priya|      4|  1|     IT| 230098456|
|  2|    Ram|      2|  2|Finance|  78374874|
|  2|    Ram|      2|  3|     HR|3435543634|
|  3|  Priya|      4|  2|Finance|  78374874|
|  3|  Priya|      4|  3|     HR|3435543634|
+---+-------+-------+---+-------+----------+



In [30]:
# Left Anti Join -->
# Purpose: Returns rows from the left DataFrame that have no match in the right DataFrame - essentially the opposite of left semi.
# Columns returned: Only columns from the left DataFrame.
left_anti_join_df = emp_df.join(dept_df, emp_df.dept_id == dept_df.id, 'left_anti')
left_anti_join_df.show()


+---+-----+-------+
| id| name|dept_id|
+---+-----+-------+
|  3|Priya|      4|
+---+-----+-------+



In [31]:
# Left Semi Join -->
# Purpose: Returns rows from the left DataFrame that have a match in the right DataFrame.
# Columns returned: Only columns from the left DataFrame - right DataFrame columns are never included, even though it's used for filtering.
left_semi_join_df = emp_df.join(dept_df, emp_df.dept_id == dept_df.id, 'left_semi')
left_semi_join_df.show()

+---+-------+-------+
| id|   name|dept_id|
+---+-------+-------+
|  1|Gayatri|      1|
|  2|    Ram|      2|
+---+-------+-------+



In [ ]:
# Task: Create a new DF with Columns: emp_id, emp_name, dept_id, dept_name


***select()***

Syntax: Takes column objects or column names — requires col(), expr(), or DataFrame column references for transformations.
Expression style: Programmatic — you build expressions using Column API methods (.alias(), .cast(), .substr(), etc.).
SQL strings: Can't directly write raw SQL expressions as strings unless wrapped in expr().
Readability: More verbose for complex transformations, but type-safe and IDE-friendly (autocomplete works well).

***selectExpr()***

Syntax: Takes SQL expressions directly as strings — no need for col() or expr() wrapping.
Expression style: SQL-like — you write expressions the way you would in a SQL SELECT clause.
SQL strings: Native support for raw SQL syntax, including functions, CASE WHEN, arithmetic, aliasing with AS.
Readability: More concise for SQL-familiar users, but loses some compile-time safety (errors surface at runtime).

**Types of JOINS in PySpark**

***1. Inner Join (inner)***

Returns only rows with matching keys in both DataFrames.
Default join type if how is not specified.

***2. Left Outer / Left Join (left, left_outer / leftouter)***

Returns all rows from the left DataFrame, plus matched rows from the right; unmatched right columns are null.

***3. Right Outer / Right Join (right, right_outer / rightouter)***

Returns all rows from the right DataFrame, plus matched rows from the left; unmatched left columns are null.

***4. Full Outer / Outer Join (outer, full, full_outer / fullouter)***

Returns all rows from both DataFrames; unmatched columns on either side are null.

***5. Left Semi Join (left_semi / leftsemi)***

Returns rows from the left DataFrame that have a match in the right - only left columns are returned.

***6. Left Anti Join (left_anti / leftanti)***

Returns rows from the left DataFrame that don't have a match in the right - only left columns are returned.

***7. Cross Join (cross)***

Returns the Cartesian product - every row of left paired with every row of right, no join key needed.

***8. Self Join***

Not a distinct how value - it's joining a DataFrame to itself, typically by aliasing it twice.